In [5]:
from collections import defaultdict
from matplotlib import pyplot as plt
from parselog import *
from logtools import *
import gc
import itertools
import math
import multiprocessing
import numpy as np
import os
import pickle
import sys

In [6]:
BENCHMARKS = [
    "bh",
    "health",
    "perimeter",
    "tsp",
]
PREFETCHERS_LOCATIONS = [
    ("cap_chaser_allin_h_always_m_always", "l1ll"),
	("cap_chaser_allin_h_50_m_50", "l1ll"),
	("cap_chaser_allin_h_50_m_always", "l1ll"),
	("cap_chaser_allin_h_75_m_75", "l1ll"),
	("cap_chaser_allin_h_75_m_50", "l1ll"),
	("cap_chaser_allin_h_75_m_always", "l1ll"),
	("cap_chaser_allin_h_never_m_never", "l1ll"),
	("cap_chaser_allin_h_never_m_75", "l1ll"),
	("cap_chaser_allin_h_never_m_50", "l1ll"),
	("cap_chaser_allin_h_never_m_always", "l1ll"),
]
MINRVFI = 50000
MAXRVFI = None

def loadLoadWaitData(run, prefetcherLocation, benchmark, minRvfi, maxRvfi):
    _, _, _, lp = loadPrefetcherData(run, prefetcherLocation, benchmark, minRvfi, maxRvfi)
    return run, prefetcherLocation, benchmark, np.average(lp.dists[CRqCreationLine]["demandHitCycles"])

loadWaitReds = {}
if os.path.isfile("results/pickles/allinData/load-wait.pickle"):
    print("Loading from pickle...")
    with open("results/pickles/allinData/load-wait.pickle", "rb") as fp:
        loadWaitReds = pickle.load(fp)
    print("... finished loading from pickle")
else:
    tuples = list(itertools.product(["cap_chaser_allin"], PREFETCHERS_LOCATIONS, BENCHMARKS))

    results = None
    with multiprocessing.Pool(processes=16) as pool:
        maxRvfis = pool.starmap(getMaxRVFI, tuples)
        minMaxRvfisForBenchmark = defaultdict(list)
        for run, prefetcherLocation, benchmark, maxRvfi in maxRvfis:
            minMaxRvfisForBenchmark[benchmark].append(maxRvfi)
        minMaxRvfisForBenchmark = {k: min(min(l), MAXRVFI or float("inf")) for (k, l) in minMaxRvfisForBenchmark.items()}
        print()
        for benchmark, minRvfi in minMaxRvfisForBenchmark.items():
            print(f"{benchmark}: {minRvfi}")
        print()
        sys.stdout.flush()
        tuplesWithRvfi = [t + (MINRVFI,minMaxRvfisForBenchmark[t[-1]],) for t in tuples]
        results = pool.starmap(loadLoadWaitData, tuplesWithRvfi)

    loadWaits = defaultdict(list)
    for run, prefetcherLocation, benchmark, lw in results:
        loadWaits[prefetcherLocation].append(lw)

    loadWaitReds = defaultdict(list)
    BASELINE = ("cap_chaser_allin_h_never_m_never", "l1ll")
    for prefetcherLocation in PREFETCHERS_LOCATIONS:
        if prefetcherLocation != BASELINE:
            for i in range(len(BENCHMARKS)):
                loadWaitReds[prefetcherLocation].append(
                    1 - (loadWaits[prefetcherLocation][i]/loadWaits[BASELINE][i])
                )
    loadWaitReds[BASELINE] = [0, 0, 0, 0]
    loadWaitReds = toDict(loadWaitReds)

    with open("results/pickles/allinData/load-wait.pickle", "wb") as fp:
        pickle.dump(loadWaitReds, fp)


Scanning cap_chaser_allin-cap_chaser_allin_h_always_m_always-l1ll/bh...
Scanning cap_chaser_allin-cap_chaser_allin_h_always_m_always-l1ll/perimeter...
Scanning cap_chaser_allin-cap_chaser_allin_h_always_m_always-l1ll/tsp...
Scanning cap_chaser_allin-cap_chaser_allin_h_always_m_always-l1ll/health...
Scanning cap_chaser_allin-cap_chaser_allin_h_50_m_50-l1ll/bh...
Scanning cap_chaser_allin-cap_chaser_allin_h_50_m_50-l1ll/health...
Scanning cap_chaser_allin-cap_chaser_allin_h_50_m_50-l1ll/perimeter...
Scanning cap_chaser_allin-cap_chaser_allin_h_50_m_50-l1ll/tsp...
Scanning cap_chaser_allin-cap_chaser_allin_h_50_m_always-l1ll/health...
Scanning cap_chaser_allin-cap_chaser_allin_h_50_m_always-l1ll/bh...
Scanning cap_chaser_allin-cap_chaser_allin_h_50_m_always-l1ll/perimeter...
Scanning cap_chaser_allin-cap_chaser_allin_h_50_m_always-l1ll/tsp...
Scanning cap_chaser_allin-cap_chaser_allin_h_75_m_75-l1ll/bh...
Scanning cap_chaser_allin-cap_chaser_allin_h_75_m_75-l1ll/health...
Scanning cap_cha

In [7]:
maxSpeedup = max(abs(s) for speedup in speedups.values() for s in speedup)
for pl, speedup in speedups.items():
    print(f"\n{pl}:")
    for s, b in zip(speedup, BENCHMARKS):
        if s == 0:
            color = "0 0 0"
        elif s < 0:
            v = int(0xff * (abs(s)/maxSpeedup)**0.7)
            color = f"255 {(0xff-v)} {(0xff-v)}"
        else:
            v = int(0xff * (abs(s)/maxSpeedup)**0.7)
            color = f"{(0xff-v)} 255 {(0xff-v)}"
        print(f"\t{b}: {round(s*100, 1)} :: {color}")

NameError: name 'speedups' is not defined

In [15]:
maxLWRed = max(abs(x) for lwRed in loadWaitReds.values() for x in lwRed)
for pl, lwRed in loadWaitReds.items():
    print(f"\n{pl}:")
    for x, b in zip(lwRed, BENCHMARKS):
        if x == 0:
            color = "0 0 0"
        elif x < 0:
            v = int(0xff * (abs(x)/maxLWRed))
            color = f"255 {(0xff-v)} {(0xff-v)}"
        else:
            v = int(0xff * (abs(x)/maxLWRed)**1.2)
            color = f"{(0xff-v)} 255 {(0xff-v)}"
        print(f"\t{b}: {round(x*100, 1)} :: {color}")


('cap_chaser_allin_h_always_m_always', 'l1ll'):
	bh: 7.3 :: 222 255 222
	health: 39.6 :: 0 255 0
	perimeter: 18.2 :: 155 255 155
	tsp: 24.6 :: 112 255 112

('cap_chaser_allin_h_50_m_50', 'l1ll'):
	bh: 6.8 :: 225 255 225
	health: 23.2 :: 122 255 122
	perimeter: 17.2 :: 162 255 162
	tsp: 25.9 :: 102 255 102

('cap_chaser_allin_h_50_m_always', 'l1ll'):
	bh: 6.9 :: 224 255 224
	health: 34.9 :: 36 255 36
	perimeter: 18.3 :: 155 255 155
	tsp: 25.9 :: 102 255 102

('cap_chaser_allin_h_75_m_75', 'l1ll'):
	bh: 7.1 :: 223 255 223
	health: 2.0 :: 248 255 248
	perimeter: 9.6 :: 209 255 209
	tsp: 26.2 :: 100 255 100

('cap_chaser_allin_h_75_m_50', 'l1ll'):
	bh: 6.9 :: 224 255 224
	health: 10.9 :: 202 255 202
	perimeter: 14.6 :: 178 255 178
	tsp: 26.2 :: 101 255 101

('cap_chaser_allin_h_75_m_always', 'l1ll'):
	bh: 7.0 :: 224 255 224
	health: 29.2 :: 79 255 79
	perimeter: 17.3 :: 161 255 161
	tsp: 25.5 :: 105 255 105

('cap_chaser_allin_h_never_m_75', 'l1ll'):
	bh: 5.6 :: 231 255 231
	health: 1.5 :